# Module 3: Vision Transformers in PyTorch

This notebook trains and compares two CNN-ViT hybrid models for land classification.

The baseline `model` uses the original simple Vision Transformer settings. The `model_test` model uses the required test configuration: 5 epochs, 12 attention heads, embedding dimension 768, and transformer depth 12.

## Imports and setup

Set a seed so the data split and model comparison are repeatable.

In [1]:
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 7331
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Task 1: Create `train_transform`

Training images are resized, augmented, converted to tensors, and normalized with the same values used in the original training notebook.

In [2]:
DATASET_PATH = './images_dataSAT'
if not os.path.isdir(DATASET_PATH):
    DATASET_PATH = os.path.join('.', 'AI Capstone DL Projects', 'CNN Model Development', 'images_dataSAT')

image_size = (64, 64)
batch_size = 32
num_classes = 2

train_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomRotation(40),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print(train_transform)

Compose(
    Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-40.0, 40.0], interpolation=nearest, expand=False, fill=0)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## Task 2: Create `val_transform`

Validation data uses only deterministic resizing, tensor conversion, and normalization. Random augmentation is not used during validation.

In [3]:
val_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print(val_transform)

Compose(
    Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## Prepare training and validation datasets

Use the same random split for both transformed dataset views so the models compare on identical samples.

In [4]:
augmented_dataset = datasets.ImageFolder(DATASET_PATH, transform=train_transform)
validation_dataset = datasets.ImageFolder(DATASET_PATH, transform=val_transform)

train_size = int(0.8 * len(augmented_dataset))
val_size = len(augmented_dataset) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(range(len(augmented_dataset)), [train_size, val_size], generator=split_generator)

train_dataset = torch.utils.data.Subset(augmented_dataset, train_indices.indices)
val_dataset = torch.utils.data.Subset(validation_dataset, val_indices.indices)
print('Training samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))

Training samples: 4800
Validation samples: 1200


## Task 3: Create `train_loader` and `val_loader`

The loaders use the training and validation datasets created above. Validation batches are not shuffled so evaluation order remains stable.

In [5]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))

Training batches: 150
Validation batches: 38


## Define the CNN-ViT hybrid

The CNN reduces each 64x64 image to one spatial feature token. The class token then passes through transformer blocks before classification.

In [6]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(256),
            nn.Conv2d(256, 512, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(512),
            nn.Conv2d(512, 1024, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(1024)
        )
    def forward(self, images):
        return self.features(images)

class PatchEmbed(nn.Module):
    def __init__(self, in_channels=1024, embed_dim=768):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=1)
    def forward(self, features):
        return self.projection(features).flatten(2).transpose(1, 2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
    def forward(self, tokens):
        normalized = self.norm1(tokens)
        attended = self.attention(normalized, normalized, normalized, need_weights=False)[0]
        tokens = tokens + attended
        return tokens + self.mlp(self.norm2(tokens))

class CNNViT(nn.Module):
    def __init__(self, num_classes=2, embed_dim=768, depth=6, heads=8):
        super().__init__()
        self.cnn = ConvNet()
        self.patch = PatchEmbed(1024, embed_dim)
        self.class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position = nn.Parameter(torch.randn(1, 2, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, images):
        tokens = self.patch(self.cnn(images))
        class_token = self.class_token.expand(images.size(0), -1, -1)
        tokens = torch.cat((class_token, tokens), dim=1)
        tokens = tokens + self.position[:, :tokens.size(1)]
        for block in self.blocks:
            tokens = block(tokens)
        return self.classifier(self.norm(tokens[:, 0]))

model = CNNViT(num_classes=num_classes, embed_dim=768, depth=6, heads=8).to(device)
print('Baseline model created')

Baseline model created


## Train and record validation loss and time

The helper trains any CNN-ViT configuration and records one validation loss and one elapsed time value per epoch.

In [7]:
def train_model(model_to_train, epochs, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_to_train.parameters(), lr=learning_rate)
    history = {'train_loss': [], 'val_loss': [], 'epoch_time': []}

    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        model_to_train.train()
        train_loss_total = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_to_train(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item() * images.size(0)

        model_to_train.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model_to_train(images)
                loss = criterion(outputs, labels)
                val_loss_total += loss.item() * images.size(0)

        epoch_time = time.perf_counter() - epoch_start
        train_loss = train_loss_total / len(train_loader.dataset)
        val_loss = val_loss_total / len(val_loader.dataset)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['epoch_time'].append(epoch_time)
        print(f'Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f} - val loss: {val_loss:.4f} - time: {epoch_time:.2f}s')

    history['total_time'] = sum(history['epoch_time'])
    return history

## Train the baseline `model`

Train the baseline model for five epochs so both models have the same training budget.

In [8]:
baseline_history = train_model(model, epochs=5)
print('Baseline total training time:', round(baseline_history['total_time'], 2), 'seconds')

Epoch 1/5 - train loss: 0.4268 - val loss: 0.0914 - time: 449.64s
Epoch 2/5 - train loss: 0.0891 - val loss: 0.0382 - time: 253.88s
Epoch 3/5 - train loss: 0.0492 - val loss: 0.0201 - time: 275.34s
Epoch 4/5 - train loss: 0.0769 - val loss: 0.0585 - time: 338.62s
Epoch 5/5 - train loss: 0.0704 - val loss: 0.0408 - time: 308.11s
Baseline total training time: 1625.57 seconds


## Task 4: Design and train `model_test`

The test model uses the required hyperparameters:
- `epochs = 5`
- `heads = 12`
- `embed_dim = 768`
- `depth = 12`

In [ ]:
test_epochs = 5
test_heads = 12
test_embed_dim = 768
test_depth = 12

model_test = CNNViT(
    num_classes=num_classes,
    embed_dim=test_embed_dim,
    depth=test_depth,
    heads=test_heads
).to(device)

test_history = train_model(model_test, epochs=test_epochs)
print('Test model total training time:', round(test_history['total_time'], 2), 'seconds')

## Task 5: Compare validation loss

Plot the validation loss for the baseline `model` and the larger `model_test` ViTs.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), baseline_history['val_loss'], marker='o', label='model: baseline ViT')
plt.plot(range(1, 6), test_history['val_loss'], marker='o', label='model_test: 12-head, depth-12 ViT')
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

## Task 6: Compare training time

Plot the training time for each model. The first plot compares time per epoch; the second reports total training time.

In [ ]:
epochs_axis = range(1, 6)
width = 0.35
positions = np.arange(1, 6)

plt.figure(figsize=(8, 5))
plt.bar(positions - width / 2, baseline_history['epoch_time'], width, label='model: baseline ViT')
plt.bar(positions + width / 2, test_history['epoch_time'], width, label='model_test: larger ViT')
plt.title('Training Time per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Seconds')
plt.xticks(positions, epochs_axis)
plt.legend()
plt.grid(axis='y')
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(['model', 'model_test'], [baseline_history['total_time'], test_history['total_time']], color=['steelblue', 'darkorange'])
plt.title('Total Training Time')
plt.ylabel('Seconds')
plt.show()

print('Total baseline time:', round(baseline_history['total_time'], 2), 'seconds')
print('Total model_test time:', round(test_history['total_time'], 2), 'seconds')

## Module 3 completed

The notebook creates both transforms, both dataloaders, trains the baseline and required test model, and plots validation-loss and training-time comparisons.